# Phase 0: Laya smoke test and interface check

Purpose: confirm that the pinned `laya==0.3.24` package installs and answers all three
typed question primitives (choice, score, noul) in one call, and pin down the exact
question format the installed package accepts.

The Phase 0 reproduction run (upstream fine-tune + Gate 0 evaluation) is not here; it
lives in `step5_colab_launcher.ipynb`.

## Cell A: setup

Public repo, no token needed. **Needs no GPU:** the smoke test runs on a CPU runtime.

In [ ]:
# Cell A: setup (public repo). Needs no GPU.
import os
os.chdir("/content")  # stand somewhere real (survives a deleted working directory)

from google.colab import drive
drive.mount("/content/drive")

REPO = "/content/laya-vision"
if not os.path.exists(REPO):
    !git clone -q https://github.com/savita10/laya-vision.git {REPO}
else:
    !git -C {REPO} pull -q --ff-only
os.chdir(REPO)
!git log --oneline -1

!pip install -q "transformers==4.49.0" "laya==0.3.24"

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU: fine for this notebook.")

## Smoke test

One state (a customer email) and three questions, keyed by question id, one per primitive:

- `category`: **choice**. `criteria` is a list of labels.
- `urgency`: **score**. `criteria` is a list of level descriptions, where list index = level (0-4).
- `churn`: **noul**. A statement; the answer is P(true). `criteria` is optional.

Expected answer shape: `{"model", "answers", "usage", "routing"}`, where `answers` holds one
entry per question id:

- choice: `choice` and `probabilities` per label;
- score: the expected `score` (0-4), a `legend` and `probabilities` per level;
- noul: `noul` = P(true).

Each answer also carries `confidence`, `answer_confidence` and `action.act_probability`.
`routing` is added by `Router`.

In [ ]:
import time
from laya import Router

router = Router()

state = "From: J. Carter\nSubject: Payment failed twice\nYour checkout rejected my card twice today and I was still charged a fee. Fix this or I'm cancelling."

questions = {
    "category": {"type": "choice",
                 "instructions": "Which team should handle this email?",
                 "criteria": ["billing", "technical", "refunds"]},
    "urgency":  {"type": "score",
                 "instructions": "How urgent is this email?",
                 "criteria": ["no action needed",          # level 0
                              "routine, this week",         # 1
                              "soon, within a day or two",  # 2
                              "today",                      # 3
                              "immediate escalation"]},     # 4
    "churn":    {"type": "noul",
                 "instructions": "The customer will churn."},
}

t0 = time.perf_counter()
answers = router.predict(state, questions)
print(answers)
print(f"inference: {(time.perf_counter()-t0)*1000:.0f} ms")

## Interface discovery

The published docs lagged the installed package, so the question format was taken from
the code. The validator `Agent._check_question` was the ground truth: it names the
question and what to fix for every malformed definition. The next three cells print
the validator, `Router.predict`'s docstring, and every score/criteria rule.

The interface rules found here are recorded in DECISIONS.md, "Backfill: Phase 0 Step 4
findings (2026-10-01, recorded late)".

In [ ]:
import laya.agent, inspect

# Print Agent._check_question, the question validator, wherever it sits in the installed version.
src = inspect.getsource(laya.agent).splitlines()
start = next(i for i, line in enumerate(src) if line.lstrip().startswith("def _check_question"))
indent = len(src[start]) - len(src[start].lstrip())
end = start + 1
while end < len(src) and (not src[end].strip() or len(src[end]) - len(src[end].lstrip()) > indent):
    end += 1
for i in range(start, end):
    print(f"{i + 1:4d}  {src[i]}")

In [ ]:
from laya import Router
print(inspect.getdoc(Router.predict))

In [ ]:
import laya.agent, inspect
src = inspect.getsource(laya.agent)
for i, line in enumerate(src.splitlines(), 1):
    if "score" in line.lower() and ("criteria" in line.lower() or "raise" in line.lower() or "def " in line.lower()):
        print(f"{i:4d}  {line}")

## Interface cheat-sheet

- `questions` is a **dict keyed by question id**. Each definition needs a `type`
  (`choice` / `score` / `noul`) and a non-empty `instructions`.
- **choice:** `criteria` is a list of labels (or a dict of label -> description). Labels
  are scalar, non-null and distinct; they become the answer keys.
- **score:** `criteria` is a list of level descriptions. List index = level, so give
  them in order from 0, with no null levels. The answer is the expected level.
- **noul:** `instructions` is a statement. `criteria` is optional. The answer `noul`
  is P(true).
- `Router().predict(state, questions)` routes to a checkpoint, then answers every
  question in one forward pass (`Agent.predict` is `system_one`).